<a href="https://colab.research.google.com/github/emijlop/master_thesis/blob/main/notebooks/01_bbq_rq1_internal_coherence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 · BBQ internal coherence (RQ1)

**Thesis:** *Same Category, Different Territory: Institutional and Community Framings of Social Bias in LLM Benchmarks*

**RQ1:** does each of BBQ's own categories map onto a topically coherent cluster, or scatter across unrelated ones?

Pipeline: frozen LaBSE → UMAP → HDBSCAN (Marchiori Manerba et al., 2024). Design decisions are recorded in [`docs/decision_log.md`](../docs/decision_log.md), and metric definitions in [`docs/research_design.md`](../docs/research_design.md).

| Proposal stage | Section |
|---|---|
| 1. Framing extraction | §1 |
| 2. Normalization / unit of analysis | §2 |
| 3. Embedding (frozen LaBSE) | §3 |
| 4–5. UMAP + HDBSCAN | §4 |
| 6. Coherence: (i) purity, (ii) dispersion | §5 |
| Statistical tests (template-level nulls) | §5b |
| Intersectionality | §6 |
| Sensitivity (stage 5) | §7 |
| Tentative classifier | §8 |
| 7. Interpretation | §9 |
| Hand-off to RQ2/RQ3 | §10 |
| Run manifest (hardware, versions, timings) | §11 |

In [ ]:
# Setup: works in Colab (clones the repo) and locally (pip install -e ".[pipeline]" from the repo root)
import sys, subprocess
from pathlib import Path
if "google.colab" in sys.modules and not Path("master_thesis").exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/emijlop/master_thesis.git"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "master_thesis[pipeline]"], check=True)
for p in [Path("master_thesis/src"), Path("../src")]:
    if p.exists(): sys.path.insert(0, str(p.resolve()))

In [ ]:
import warnings
import numpy as np, pandas as pd
from benchbias import SEED, ENCODER, ROBUST_ENCODER
from benchbias import bbq, metrics, cluster, probe, plots, schema, stats
from benchbias.embed import embed, get_encoder
from benchbias.runlog import RunLog

pd.set_option("display.max_colwidth", 200)
warnings.filterwarnings("ignore", category=FutureWarning)
OUT = Path("outputs"); OUT.mkdir(exist_ok=True)
B_PERM = 999          # permutations / bootstrap draws for the statistical tests (§5b); 199 for a quick run
RUN = "default"       # MODIFY per setup in the run plan, e.g. "mpnet", "sofa_settings"; tags every output file
log = RunLog(f"01_bbq_rq1_{RUN}", config={"encoder": ENCODER, "seed": SEED, "B_perm": B_PERM})
log.data["hardware"]

## 1. Framing extraction: what BBQ claims vs. how its items are distributed

BBQ measures *attested* stereotypes against protected classes along nine U.S.-centric axes, plus two intersectional ones. Item counts per category reflect **how many names or group labels were substituted into each template**, not how many distinct stereotypes were written. `share_of_items` sets a category's weight in the aggregate score; `share_of_templates` shows its conceptual breadth.

In [ ]:
raw = bbq.load_bbq_raw()
print(raw.shape)
framing = bbq.framing_table(raw)
framing.to_csv(OUT / "bbq_framing_table.csv")
framing

## 2. Unit of analysis: two views

**Unit:** ambiguous context + negative-polarity question. This pairs the scenario BBQ's authors believe triggers the stereotype with the attribute the stereotype attaches to. The disambiguating sentence encodes the *answer*, not the *construct*, so it is left out.

- **`unmasked`**: identity terms kept. *Who* does the category target? At most `K_PER_TEMPLATE` instances per template.
- **`masked`**: target entities → *Person A / Person B*. *What* is said about the category? De-duplicating on masked text collapses name substitutions, which amounts to template-level normalization.

If categories are coherent only when unmasked, they are held together by group labels, not by a shared stereotype construct.

In [ ]:
K_PER_TEMPLATE = 6
D = bbq.build_units(raw, k_per_template=K_PER_TEMPLATE, seed=SEED)

print("masking coverage (both targets found):", (D["masked"]["mask_hits"] >= 2).mean().round(3))
display(D["masked"].loc[D["masked"]["mask_hits"] < 2, ["category", "context", "ents"]].head(5))  # inspect failures

pd.DataFrame({v: D[v]["category"].value_counts() for v in D}).join(
    raw.groupby("category")["template_id"].nunique().rename("templates"))

In [ ]:
# Masked texts that occur verbatim under more than one category. If e.g. Race_x_gender and Race_ethnicity
# share identical scenarios once the names are removed, the category is carried only by the substituted label.
bbq.shared_texts(D["masked"])

## 3. Embedding (frozen LaBSE)

LaBSE stays frozen, so whatever structure appears comes from the pretrained space. Embeddings are cached in `outputs/`.
*Caveat:* LaBSE was trained for translation-pair retrieval, not semantic similarity, so §7 repeats the analysis with an STS-trained multilingual encoder.

In [ ]:
with log.step("embed"):
    E = {v: embed(D[v]["text"], ENCODER, cache=OUT / f"emb_bbq_{v}_LaBSE.npy") for v in D}
log.data["config"].update(K_PER_TEMPLATE=K_PER_TEMPLATE, n_items={v: len(D[v]) for v in D})
{v: e.shape for v, e in E.items()}

## 4. UMAP → HDBSCAN

BERTopic is used only as a wrapper, for c-TF-IDF keywords that help with qualitative reading (§9). All components are explicit and seeded.

In [ ]:
MIN_CLUSTER = 10
log.data["config"]["min_cluster"] = MIN_CLUSTER
models = {}
for v in D:
    tm = cluster.make_topic_model(get_encoder(ENCODER), min_cluster=MIN_CLUSTER, seed=SEED)
    topics, _ = tm.fit_transform(D[v]["text"].tolist(), embeddings=E[v])
    D[v]["cluster"] = topics
    models[v] = tm
    t = np.asarray(topics)
    print(f"{v}: {t.max() + 1} clusters | noise share {np.mean(t == -1):.3f}")
    log.add_result(f"n_clusters_{v}", int(t.max() + 1)); log.add_result(f"noise_{v}", float(np.mean(t == -1)))

## 5. RQ1: coherence metrics

| Proposal metric | Operationalization |
|---|---|
| (i) category purity per cluster | **homogeneity**; dominant-category share per cluster |
| (ii) category dispersion | **completeness**; clusters needed to cover 80% of a category; normalized entropy |

Reference points: a **permutation null** (floor) and the **template** labels. If agreement with templates is much higher than with categories, the clusters are recovering the authors' writing units, not a shared construct.

In [ ]:
for v in D:
    print(f"\n=== {v} ===")
    summ = metrics.rq1_summary(D[v], seed=SEED)
    summ.to_csv(OUT / f"rq1_summary_{v}_{RUN}.csv")
    display(summ)

In [ ]:
per_cat = pd.concat({v: metrics.per_category(D[v]) for v in D}, axis=1)
per_cat.to_csv(OUT / f"rq1_per_category_{RUN}.csv")
per_cat

In [ ]:
clu = metrics.per_cluster(D["masked"])
mixed = clu[clu["purity"] < 0.7].join(models["masked"].get_topic_info().set_index("Topic")["Name"])
print("Mixed clusters (purity < 0.7), masked view: scenarios shared across BBQ categories")
mixed

In [ ]:
from umap import UMAP
for v in D:
    plots.category_cluster_heatmap(D[v], f"BBQ category × cluster ({v}, LaBSE, min_cluster={MIN_CLUSTER})",
                                   OUT / f"heatmap_bbq_{v}.png")
xy = UMAP(n_neighbors=15, n_components=2, min_dist=0.1, metric="cosine", random_state=SEED).fit_transform(E["masked"])
plots.small_multiples(xy, D["masked"]["category"], "Where each BBQ category sits in LaBSE space (masked)",
                      OUT / "map_bbq_masked.png");

## 5b. Statistical tests for RQ1

The descriptive metrics above need a null hypothesis. BBQ's unit of independent information is the **template**: name substitutions of one template are near-copies. Every null distribution therefore reassigns whole templates, never single items.

- **H0 (cluster test):** given the template structure, category labels carry no information about cluster membership. Statistic: AMI(clusters, categories). The template-level null reassigns categories to whole templates, preserving the number of templates per category. The item-level null (free shuffling) is reported only to show how badly it overstates significance.
- **H0 (clustering-free test):** a category's items are no more similar to each other than to the rest, beyond template similarity. Statistic: within- minus between-category mean cosine similarity in the original 768-dimensional space, so it doesn't depend on UMAP or HDBSCAN settings. Per-category p-values are BH-corrected.
- **Effect sizes:** 95% percentile CIs from resampling templates (cluster bootstrap).

Reading: categories are coherent if both tests reject under the template-level null. Coherence that exists only in the unmasked view points to identity terms, not stereotype content.

In [ ]:
tests_rq1 = {}
for v in D:
    with log.step(f"tests_{v}"):
        t = stats.rq1_permutation_test(D[v]["category"], D[v]["cluster"], D[v]["template_id"], B=B_PERM, seed=SEED)
        gap = stats.rq1_gap_test(E[v], D[v]["category"].to_numpy(), D[v]["template_id"].to_numpy(), B=B_PERM, seed=SEED)
        ci = stats.bootstrap_by_group(lambda i: stats.coherence_gap(E[v][i], D[v]["category"].to_numpy()[i])[0],
                                      D[v]["template_id"].to_numpy(), B=B_PERM, seed=SEED)
    tests_rq1[v] = {**t, **{f"gap_{k}": val for k, val in ci.items()}}
    gap.to_csv(OUT / f"rq1_gap_test_{v}_{RUN}.csv")
    log.add_result(f"rq1_tests_{v}", tests_rq1[v])
    print(f"\n=== {v} ===")
    display(gap)
pd.DataFrame(tests_rq1).T.round(4).to_csv(OUT / f"rq1_tests_{RUN}.csv")
pd.DataFrame(tests_rq1).T.round(4)

## 6. Intersectionality probe

The theory chapter draws on Crenshaw and Lugones: axes are *co-constituted*. Does BBQ's operationalization reflect that?
- **Co-constitutive reading:** Race×Gender / Race×SES items form their own clusters.
- **Additive reading:** they fall inside single-axis clusters, most likely Race_ethnicity.

In [ ]:
{v: metrics.intersection_placement(D[v], bbq.INTERSECTIONAL) for v in D}

## 7. Sensitivity and robustness

Metrics are recomputed across `min_cluster_size` × UMAP seeds and reported as ranges, plus seed stability (mean pairwise ARI). The same check is repeated with a second encoder. A finding enters the thesis only if it holds across this grid.

In [ ]:
with log.step("sensitivity"):
    sens = {v: cluster.sweep(D[v]["category"], E[v]) for v in D}
for v in D:
    sens[v].to_csv(OUT / f"rq1_sensitivity_{v}_{RUN}.csv")
sens["masked"]

In [ ]:
E_rob = embed(D["masked"]["text"], ROBUST_ENCODER, cache=OUT / "emb_bbq_masked_mpnet.npy")
rob = cluster.sweep(D["masked"]["category"], E_rob, sizes=(10, 25), seeds=(0, 1))
rob.to_csv(OUT / f"rq1_sensitivity_masked_mpnet_{RUN}.csv")
rob

## 8. Linear probe (proposal §4.2.1, tentative)

Clustering asks whether categories form *regions*; a probe asks whether they are *separable* at all. The confusion matrix shows which categories BBQ's own content fails to distinguish, which is a structural-validity signal. CV is **grouped by template**, and the gap from a random split shows how much of the "category signal" is really template signal.

In [ ]:
cms, probes = {}, {}
for v in D:
    single = ~D[v]["intersectional"].to_numpy()
    scores, cms[v], probes[v] = probe.probe(E[v][single], D[v].loc[single, "category"],
                                            D[v].loc[single, "template_id"], seed=SEED)
    print(v, {k: round(s, 3) for k, s in scores.items()})
    log.add_result(f"probe_{v}", scores)
    cms[v].to_csv(OUT / f"probe_confusion_{v}_{RUN}.csv")
cms["masked"]

In [ ]:
# Intersectional items, classified by a probe trained only on single-axis categories
for v in D:
    m = D[v]["intersectional"].to_numpy()
    print(f"\n[{v}]")
    display(pd.crosstab(D[v].loc[m, "category"], probes[v].predict(E[v][m]), normalize="index").round(3))

## 9. Interpretation scaffold (proposal stage 7)

| Observation | Evidence | Validity facet (Wallach et al., 2024) | Harm type (Shelby et al., 2023) |
|---|---|---|---|
| Item weight ≠ conceptual breadth | §1 | substantive | — |
| Coherent only when unmasked → held together by labels | §5 masked vs. unmasked | substantive | stereotyping / reification |
| Mixed clusters across categories | §5 `mixed` | structural | — |
| Intersectional items placed in a single parent axis | §6, §8 | structural (additive vs. co-constitutive) | erasure |
| Categories confused by the probe | §8 | structural | — |

In [ ]:
models["masked"].get_topic_info().head(25)

## 10. Export in the common schema (for RQ2/RQ3)

See [`docs/crosswalk.md`](../docs/crosswalk.md). The language and format confounds for RQ3 are described in [`docs/research_design.md`](../docs/research_design.md).

In [ ]:
for v in D:
    schema.to_schema(D[v], "BBQ").to_parquet(OUT / f"bbq_{v}.parquet", index=False)
    np.save(OUT / f"bbq_{v}_labse.npy", E[v])
    # cluster assignments, same row order as the embeddings; used by notebook 02 (identity probes)
    D[v][["template_id", "category", "intersectional", "text", "cluster"]].to_parquet(
        OUT / f"bbq_{v}_clusters.parquet", index=False)
sorted(p.name for p in OUT.iterdir())

## 11. Run manifest

Hardware, package versions, git commit, configuration, timings and headline results of this run, saved as `outputs/run_01_bbq_rq1_<RUN>_<timestamp>.json`. Send this file together with the CSVs when sharing results.

In [ ]:
path = log.save(OUT)
print(path)
log.data["hardware"], log.data["timings_s"]